# XModalix3D
## Full run report

This notebook is the XModalix3D counterpart of the `imagix3d_full_report_template.ipynb`.

It is intended to provide a reproducible report of a trained **cross-modal 3D VAE** run, with particular emphasis on:

- the modality-specific sub-VAEs and their optimization settings,
- XModalix-specific loss components,
- within-modality 3D reconstruction quality,
- cross-modal translation quality,
- latent-space structure,
- quantitative cross-modal alignment using **modality linear probing** and **matched/unmatched patient distances**,
- optional downstream embedding evaluation.

The report deliberately distinguishes two aspects of cross-modal alignment:

1. **Modality invariance**: whether modality identity can still be linearly decoded from the latent representation.
2. **Patient-specific correspondence**: whether embeddings of the same patient are closer across modalities than embeddings of different patients.

For model-development decisions, use the **validation split**. The **test split should be retained for final reporting rather than model selection**.


In [ ]:
# Parameters

# These values can either be filled in here or supplied as environment variables
# by the report-generating job.
RESULT_DIR = ""
PKL_PATH = ""
REPORT_DIR = ""

# Optional downstream embedding evaluation.
EVALUATION_PARAMS = ["median_split"]

# Report controls
PRINT_FULL_MODELS = True
N_RECON_SAMPLES = 3
N_TRANSLATION_SAMPLES = 3
TRANSLATION_GROUP_PARAM = None
MAX_TRANSLATION_VOXELS = 10000
RECON_METRIC_MAX_SAMPLES = None   # None = all available samples
DIVERSITY_MAX_SAMPLES = 20
MEAN_MAP_MAX_SAMPLES = 30
DPI = 300

ALIGNMENT_REPRESENTATIONS = "z"
LINEAR_PROBE_C = 1.0
LINEAR_PROBE_CV_SPLITS = 5

### Loading the pipeline

The same CPU-safe loading approach as in the Imagix3D report is used here. This is useful when the serialized pipeline contains tensors that were originally created on CUDA.

After loading, the report explicitly replaces the evaluator and visualizer with `XModalix3DEvaluator` and `XModal3DVisualizer` so that the new XModalix3D-specific alignment evaluation and 3D visualization methods are available.


In [ ]:
from pathlib import Path
import io
import json
import os
from enum import Enum

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

import autoencodix as acx

from autoencodix.evaluate._xmodalix3d_evaluator import XModalix3DEvaluator
from autoencodix.visualize._xmodal3d_visualizer import XModal3DVisualizer
from autoencodix.utils._utils import find_translation_keys

RESULT_DIR = os.environ.get("RESULT_DIR", RESULT_DIR)
PKL_PATH = os.environ.get("PKL_PATH", PKL_PATH)
REPORT_DIR = os.environ.get("REPORT_DIR", REPORT_DIR)

if not PKL_PATH:
    raise ValueError("PKL_PATH is empty. Set it above or provide it as an environment variable.")

result_dir = Path(RESULT_DIR).resolve() if RESULT_DIR else Path.cwd().resolve()
pkl_path = Path(PKL_PATH).resolve()
report_dir = Path(REPORT_DIR).resolve() if REPORT_DIR else (result_dir / "xmodalix3d_report")

report_dir.mkdir(parents=True, exist_ok=True)

print("Result directory:", result_dir)
print("Pickle path:", pkl_path)
print("Report directory:", report_dir)

if not pkl_path.is_file():
    raise FileNotFoundError(f"Pickle file does not exist: {pkl_path}")

_original_load_from_bytes = torch.storage._load_from_bytes

def _load_from_bytes_cpu(bytes_object):
    return torch.load(
        io.BytesIO(bytes_object),
        map_location=torch.device("cpu"),
    )

torch.storage._load_from_bytes = _load_from_bytes_cpu

try:
    xmodalix3d_loaded = acx.XModalix3D.load(str(pkl_path))
finally:
    torch.storage._load_from_bytes = _original_load_from_bytes

result = xmodalix3d_loaded.result

print("Loaded XModalix3D pipeline")
print("Evaluator:", type(xmodalix3d_loaded.evaluator).__name__)
print("Visualizer:", type(xmodalix3d_loaded.visualizer).__name__)

### Helper functions for stored epochs and array conversion

In [ ]:
def _nonempty(value):
    if value is None:
        return False
    if isinstance(value, dict):
        return len(value) > 0
    try:
        return len(value) > 0
    except TypeError:
        return True


def stored_epochs_for_split(store, split):
    epochs = []
    data = getattr(store, "_data", {})
    if isinstance(data, dict):
        for epoch, split_data in data.items():
            if epoch == -1:
                continue
            if isinstance(split_data, dict) and _nonempty(split_data.get(split)):
                epochs.append(epoch)
    return sorted(set(epochs))


def resolve_epoch(store, split):
    # Prefer a final prediction stored at epoch=-1.
    try:
        value = store.get(epoch=-1, split=split)
        if _nonempty(value):
            return -1
    except Exception:
        pass

    epochs = stored_epochs_for_split(store, split)
    if not epochs:
        raise ValueError(
            f"No stored data found for split={split!r} in {type(store).__name__}."
        )
    return max(epochs)


def to_numpy(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def make_json_safe(obj):
    if isinstance(obj, Enum):
        return obj.value
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_json_safe(v) for v in obj]
    return obj


def config_to_dict(config):
    if hasattr(config, "model_dump"):
        cfg = config.model_dump(mode="python")
    elif hasattr(config, "dict"):
        cfg = config.dict()
    else:
        raise TypeError(f"Unsupported config type: {type(config)}")
    return make_json_safe(cfg)


def extract_dataset_array(dataset, index):
    item = dataset[index]

    if isinstance(item, (tuple, list)):
        if len(item) < 2:
            raise ValueError("Dataset item tuple/list does not contain image data.")
        value = item[1]
    else:
        value = item

    return to_numpy(value).squeeze().astype(np.float32)


### Dataset overview

In [ ]:
import numpy as np
import pandas as pd

dataset_rows = []

for split_name in ["train", "valid", "test"]:
    split_data = getattr(result.datasets, split_name, None)
    if split_data is None:
        continue

    is_fully_paired = getattr(split_data, "is_fully_paired", np.nan)
    paired_ids = getattr(split_data, "paired_sample_ids", [])
    unpaired_ids = getattr(split_data, "unpaired_sample_ids", [])

    if not hasattr(split_data, "datasets"):
        continue

    for modality, ds in split_data.datasets.items():
        shape = None
        if len(ds) > 0:
            try:
                shape = tuple(ds[0][1].shape)
            except Exception:
                pass

        dataset_rows.append(
            {
                "split": split_name,
                "modality": modality,
                "n_samples": len(ds),
                "sample_shape": shape,
                "split_is_fully_paired": is_fully_paired,
                "n_paired_samples_in_split": len(paired_ids),
                "n_unpaired_samples_in_split": len(unpaired_ids),
            }
        )

dataset_overview = pd.DataFrame(dataset_rows)
display(dataset_overview)


print("\nPaired-sample overview:")
for split_name in ["train", "valid", "test"]:
    split_data = getattr(result.datasets, split_name, None)
    if split_data is None:
        continue
    print(
        f"{split_name}: "
        f"{len(getattr(split_data, 'paired_sample_ids', []))} paired, "
        f"{len(getattr(split_data, 'unpaired_sample_ids', []))} unpaired"
    )


### Modality-specific VAE models

In [ ]:
models = result.model

if not isinstance(models, dict):
    raise TypeError(
        "Expected XModalix3D result.model to contain a dictionary of modality-specific models."
    )

model_rows = []

for modality, model in models.items():
    module = model.module if hasattr(model, "module") else model

    try:
        n_parameters = int(sum(p.numel() for p in module.parameters()))
        n_trainable = int(sum(p.numel() for p in module.parameters() if p.requires_grad))
    except Exception:
        n_parameters = np.nan
        n_trainable = np.nan

    model_rows.append(
        {
            "modality": modality,
            "model_class": type(module).__name__,
            "n_parameters": n_parameters,
            "n_trainable_parameters": n_trainable,
        }
    )

    print(f"\n{modality}: {type(module).__name__}")
    if PRINT_FULL_MODELS:
        display(module)

model_overview = pd.DataFrame(model_rows)
display(model_overview)

### Configuration overview

In [ ]:
cfg = config_to_dict(xmodalix3d_loaded.config)

print(json.dumps(cfg, indent=2, ensure_ascii=False))

with open(report_dir / "config.json", "w", encoding="utf-8") as handle:
    json.dump(cfg, handle, indent=2, ensure_ascii=False)


#### Effective modality-specific optimization parameters

XModalix3D can use modality-specific `learning_rate`, `weight_decay`, and `beta` values. If a value is absent in a modality's `DataInfo`, the corresponding global value is used as fallback.

The adversarial classifier remains a shared cross-modal component and therefore uses the global learning rate and weight decay.


In [ ]:
config = xmodalix3d_loaded.config
effective_hparam_rows = []

for simple_name, info in config.data_config.data_info.items():
    if getattr(info, "data_type", None) == "ANNOTATION":
        continue

    local_lr = getattr(info, "learning_rate", None)
    local_wd = getattr(info, "weight_decay", None)
    local_beta = getattr(info, "beta", None)
    local_pretrain = getattr(info, "pretrain_epochs", None)

    effective_hparam_rows.append(
        {
            "modality": simple_name,
            "learning_rate_local": local_lr,
            "learning_rate_effective": (
                local_lr if local_lr is not None else config.learning_rate
            ),
            "weight_decay_local": local_wd,
            "weight_decay_effective": (
                local_wd if local_wd is not None else config.weight_decay
            ),
            "beta_local": local_beta,
            "beta_effective": (
                local_beta if local_beta is not None else config.beta
            ),
            "pretrain_epochs_local": local_pretrain,
            "pretrain_epochs_effective": (
                local_pretrain
                if local_pretrain is not None
                else config.pretrain_epochs
            ),
            "translate_direction": getattr(info, "translate_direction", None),
        }
    )

effective_hparams = pd.DataFrame(effective_hparam_rows)
display(effective_hparams)

print(
    "\nCross-modal loss weights:",
    {
        "gamma": getattr(config, "gamma", None),
        "delta_pair": getattr(config, "delta_pair", None),
        "delta_class": getattr(config, "delta_class", None),
    },
)

effective_hparams.to_csv(
    report_dir / "effective_modality_hyperparameters.csv",
    index=False,
)


### Translation direction and stored XModalix3D outputs

In [ ]:
trained_modalities = list(models.keys())

translation_keys = find_translation_keys(
    config=xmodalix3d_loaded.config,
    trained_modalities=trained_modalities,
)

from_key = translation_keys["from"]
to_key = translation_keys["to"]

print(f"Configured translation: {from_key} -> {to_key}")

for split in ["train", "valid", "test"]:
    try:
        epoch = resolve_epoch(result.reconstructions, split)
        reconstruction_dict = result.reconstructions.get(
            epoch=epoch,
            split=split,
        )
        print(
            f"{split}: reconstruction epoch={epoch}, "
            f"keys={list(reconstruction_dict.keys()) if isinstance(reconstruction_dict, dict) else type(reconstruction_dict)}"
        )
    except Exception as exc:
        print(f"{split}: no reconstruction output available ({exc})")

print("\nAvailable sub-loss terms:")
try:
    print(list(result.sub_losses.keys()))
except Exception as exc:
    print("Could not list sub-loss terms:", exc)


### Latent representations

The following cell exports the sampled latent vectors $z$ for each available split and modality.


In [ ]:
latent_summary_rows = []

representation_stores = {
    "z": result.latentspaces,
}

for split in ["train", "valid", "test"]:
    for representation_name, store in representation_stores.items():
        try:
            epoch = resolve_epoch(store, split)
            values_by_modality = store.get(epoch=epoch, split=split)
        except Exception as exc:
            print(f"Skipping {representation_name}, {split}: {exc}")
            continue

        if not isinstance(values_by_modality, dict):
            continue

        try:
            sample_ids_by_modality = result.sample_ids.get(
                epoch=epoch,
                split=split,
            )
        except Exception:
            sample_ids_by_modality = {}

        for modality, values in values_by_modality.items():
            arr = to_numpy(values)

            if arr.ndim == 1:
                arr = arr.reshape(-1, 1)
            elif arr.ndim > 2:
                arr = arr.reshape(arr.shape[0], -1)

            ids = None
            if isinstance(sample_ids_by_modality, dict):
                ids = sample_ids_by_modality.get(modality)

            frame = pd.DataFrame(
                arr,
                columns=[
                    f"{representation_name}_{i}"
                    for i in range(arr.shape[1])
                ],
            )

            if ids is not None and len(ids) == len(frame):
                frame.insert(0, "sample_id", np.asarray(ids).astype(str))

            safe_modality = modality.replace(".", "_")
            frame.to_csv(
                report_dir
                / f"{representation_name}_{safe_modality}_{split}_epoch_{epoch}.csv",
                index=False,
            )

            numeric = frame.select_dtypes(include=np.number)
            desc = numeric.describe().T

            for latent_dim, row in desc.iterrows():
                latent_summary_rows.append(
                    {
                        "split": split,
                        "epoch": epoch,
                        "representation": representation_name,
                        "modality": modality,
                        "latent_dim": latent_dim,
                        "mean": row.get("mean"),
                        "std": row.get("std"),
                        "min": row.get("min"),
                        "median": row.get("50%"),
                        "max": row.get("max"),
                    }
                )

latent_summary = pd.DataFrame(latent_summary_rows)
display(latent_summary.head(20))

latent_summary.to_csv(
    report_dir / "latent_values_summary.csv",
    index=False,
)

print("Saved modality-specific latent values and summaries to:", report_dir)


### XModalix3D loss overview

Unlike a single VAE, XModalix3D combines multiple loss components:

- modality-specific reconstruction and variational losses,
- adversarial loss,
- paired loss,
- optionally class-based loss,
- the internal modality-classifier loss used during adversarial training.

The table below exports all stored loss curves in long format so that the exact contribution of every loss component remains available.


In [ ]:
def _curve_rows(values, split, loss_term):
    rows = []

    if values is None:
        return rows

    if isinstance(values, pd.Series):
        iterator = values.items()
    elif isinstance(values, dict):
        iterator = values.items()
    else:
        arr = np.asarray(values).squeeze()
        if arr.ndim == 0:
            iterator = [(0, arr)]
        else:
            iterator = enumerate(arr)

    for epoch, value in iterator:
        try:
            scalar = float(np.asarray(value).squeeze())
        except Exception:
            continue

        rows.append(
            {
                "epoch": int(epoch),
                "display_epoch": int(epoch) + 1,
                "split": split,
                "loss_term": loss_term,
                "value": scalar,
            }
        )

    return rows


loss_rows = []

for split in ["train", "valid"]:
    try:
        total_values = result.losses.get(split=split)
        loss_rows.extend(
            _curve_rows(total_values, split, "total_loss")
        )
    except Exception:
        pass

try:
    sub_loss_keys = list(result.sub_losses.keys())
except Exception:
    sub_loss_keys = []

for key in sub_loss_keys:
    try:
        dynamics = result.sub_losses.get(key)
    except Exception:
        continue

    for split in ["train", "valid"]:
        try:
            values = dynamics.get(split=split)
        except Exception:
            continue

        loss_rows.extend(
            _curve_rows(values, split, str(key))
        )

losses_long = pd.DataFrame(loss_rows)

if losses_long.empty:
    print("No stored loss curves found.")
else:
    final_loss_table = (
        losses_long
        .sort_values("epoch")
        .groupby(["split", "loss_term"], as_index=False)
        .tail(1)
        .sort_values(["split", "loss_term"])
        .reset_index(drop=True)
    )

    display(final_loss_table)

    losses_long.to_csv(
        report_dir / "losses_long.csv",
        index=False,
    )
    final_loss_table.to_csv(
        report_dir / "losses_final_values.csv",
        index=False,
    )


### Optional downstream embedding evaluation

As in the XModalix tutorial, each modality-specific embedding is evaluated separately. This section is disabled by default because it can rerun a complete downstream ML evaluation.

For classification, the example below uses standardized logistic regression and compares the learned embeddings with PCA, UMAP, and randomly selected features.


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000),
)

xmodalix3d_loaded.evaluate(
    params=EVALUATION_PARAMS,
    reference_methods=["PCA", "UMAP"],
    ml_model_class=clf,
)

print("Embedding evaluation completed.")

In [ ]:
for param in EVALUATION_PARAMS:
    
    xmodalix3d_loaded.visualizer.show_evaluation(
        param=param,
        metric="roc_auc_ovo",
    )
    plt.gcf().savefig(
        report_dir / f"embedding_evaluation_{param}.pdf",
        bbox_inches="tight",
    )

## 3D reconstruction and translation quality

XModalix3D trains one VAE per modality. We therefore inspect both:

- **within-modality reconstruction** for every modality,
- **cross-modal translation** for the configured source → target direction.

For the target modality, the reference reconstruction is generated from the target modality's own latent representation, while the translation is generated by encoding the source modality and decoding through the target decoder.


### Cross-modal translation grid

In [ ]:
xmodalix3d_loaded.visualizer.show_image_translation(
    result=result,
    from_key=from_key,
    to_key=to_key,
    n_sample_per_class=N_TRANSLATION_SAMPLES,
    param=TRANSLATION_GROUP_PARAM,
    split="test",
)
plt.gcf().savefig(
    report_dir / "translation_grid_test.pdf",
    bbox_inches="tight",
)

### 2D comparison of original and translated target volumes

This is the 3D-compatible counterpart of the XModalix tutorial's `show_2D_translation()`. Complete 3D volumes are too large to flatten directly, so `XModal3DVisualizer` uniformly subsamples voxel positions before applying the dimensionality reduction.


In [ ]:
xmodalix3d_loaded.visualizer.show_2D_translation(
    result=result,
    translated_modality=to_key,
    split="test",
    param=TRANSLATION_GROUP_PARAM,
    reducer="PCA",
    max_voxels=MAX_TRANSLATION_VOXELS,
)
plt.gcf().savefig(
    report_dir / "translation_pca_original_vs_translated.pdf",
    bbox_inches="tight",
)

### Quantitative reconstruction and translation metrics per sample

MSE, MAE, RMSE and percentile errors are calculated voxel-wise.

For within-modality reconstructions, each modality is compared with its own VAE reconstruction. For the configured target modality, the report additionally compares the original target volume with both the XModalix3D reference reconstruction and the cross-modal translation.


In [ ]:
def compute_output_metrics(
    result,
    *,
    split,
    modality,
    output_key,
    output_type,
    max_samples=None,
):
    split_data = getattr(result.datasets, split, None)
    if split_data is None or not hasattr(split_data, "datasets"):
        return []

    if modality not in split_data.datasets:
        return []

    dataset = split_data.datasets[modality]

    try:
        epoch = resolve_epoch(result.reconstructions, split)
        output_dict = result.reconstructions.get(
            epoch=epoch,
            split=split,
        )
        id_dict = result.sample_ids.get(
            epoch=epoch,
            split=split,
        )
    except Exception:
        return []

    if not isinstance(output_dict, dict) or output_key not in output_dict:
        return []

    outputs = to_numpy(output_dict[output_key])

    dataset_ids = np.asarray(dataset.sample_ids).astype(str)
    dataset_lookup = {
        sample_id: idx
        for idx, sample_id in enumerate(dataset_ids)
    }

    if isinstance(id_dict, dict) and output_key in id_dict:
        output_ids = np.asarray(id_dict[output_key]).astype(str)
    elif isinstance(id_dict, dict) and modality in id_dict:
        output_ids = np.asarray(id_dict[modality]).astype(str)
    else:
        output_ids = dataset_ids[: len(outputs)]

    n = min(len(outputs), len(output_ids))
    if max_samples is not None:
        n = min(n, int(max_samples))

    rows = []

    for output_idx in range(n):
        sample_id = str(output_ids[output_idx])

        if sample_id not in dataset_lookup:
            continue

        original = extract_dataset_array(
            dataset,
            dataset_lookup[sample_id],
        )
        reconstructed = np.asarray(
            outputs[output_idx]
        ).squeeze().astype(np.float32)

        if original.shape != reconstructed.shape:
            raise ValueError(
                f"Shape mismatch for {sample_id}, {output_key}: "
                f"original={original.shape}, output={reconstructed.shape}"
            )

        abs_error = np.abs(original - reconstructed)
        sq_error = (original - reconstructed) ** 2

        rows.append(
            {
                "split": split,
                "epoch": epoch,
                "modality": modality,
                "output_key": output_key,
                "output_type": output_type,
                "sample_id": sample_id,
                "mse": float(np.mean(sq_error)),
                "mae": float(np.mean(abs_error)),
                "rmse": float(np.sqrt(np.mean(sq_error))),
                "p95_abs_error": float(np.percentile(abs_error, 95)),
                "p99_abs_error": float(np.percentile(abs_error, 99)),
                "max_abs_error": float(np.max(abs_error)),
                "orig_mean": float(np.mean(original)),
                "output_mean": float(np.mean(reconstructed)),
                "orig_std": float(np.std(original)),
                "output_std": float(np.std(reconstructed)),
            }
        )

    return rows


metric_rows = []

# Within-modality reconstructions wherever they are stored.
for split in ["train", "valid", "test"]:
    for modality in trained_modalities:
        metric_rows.extend(
            compute_output_metrics(
                result,
                split=split,
                modality=modality,
                output_key=modality,
                output_type="within_modality_reconstruction",
                max_samples=RECON_METRIC_MAX_SAMPLES,
            )
        )

# Test-set target reference and translation.
try:
    test_epoch = resolve_epoch(result.reconstructions, "test")
    test_recons = result.reconstructions.get(
        epoch=test_epoch,
        split="test",
    )
except Exception:
    test_recons = {}

reference_key = f"reference_{to_key}_to_{to_key}"

if isinstance(test_recons, dict):
    if reference_key in test_recons:
        metric_rows.extend(
            compute_output_metrics(
                result,
                split="test",
                modality=to_key,
                output_key=reference_key,
                output_type="target_reference_reconstruction",
                max_samples=RECON_METRIC_MAX_SAMPLES,
            )
        )

    if "translation" in test_recons:
        metric_rows.extend(
            compute_output_metrics(
                result,
                split="test",
                modality=to_key,
                output_key="translation",
                output_type="cross_modal_translation",
                max_samples=RECON_METRIC_MAX_SAMPLES,
            )
        )

reconstruction_metrics = pd.DataFrame(metric_rows)

if reconstruction_metrics.empty:
    print("No reconstruction/translation metrics could be computed.")
else:
    display(reconstruction_metrics.head())

    display(
        reconstruction_metrics
        .groupby(["split", "modality", "output_type"])[
            ["mse", "mae", "rmse", "p95_abs_error", "orig_std", "output_std"]
        ]
        .describe()
    )

    reconstruction_metrics.to_csv(
        report_dir / "reconstruction_translation_metrics_by_sample.csv",
        index=False,
    )


### Mean target original / reference / translation / error maps

For the configured target modality, this figure summarizes the average 3D structure across a subset of test patients. It allows the reference reconstruction and cross-modal translation to be inspected side-by-side against the original target volume.


In [ ]:
def compute_mean_target_outputs(
    result,
    *,
    target_modality,
    translation_key="translation",
    n_max=30,
):
    split_data = result.datasets.test
    dataset = split_data.datasets[target_modality]

    epoch = resolve_epoch(result.reconstructions, "test")
    recon_dict = result.reconstructions.get(epoch=epoch, split="test")
    id_dict = result.sample_ids.get(epoch=epoch, split="test")

    reference_key_local = f"reference_{target_modality}_to_{target_modality}"

    if not isinstance(recon_dict, dict):
        raise ValueError("Test reconstructions are not stored as a dictionary.")

    if translation_key not in recon_dict:
        raise KeyError("No cross-modal translation is stored.")

    if reference_key_local in recon_dict:
        reference_output_key = reference_key_local
    elif target_modality in recon_dict:
        reference_output_key = target_modality
    else:
        raise KeyError("No target-modality reference reconstruction is stored.")

    dataset_ids = np.asarray(dataset.sample_ids).astype(str)
    dataset_lookup = {
        sid: idx for idx, sid in enumerate(dataset_ids)
    }

    translation_ids = np.asarray(
        id_dict[translation_key]
    ).astype(str)
    reference_ids = np.asarray(
        id_dict[reference_output_key]
    ).astype(str)

    translation_lookup = {
        sid: idx for idx, sid in enumerate(translation_ids)
    }
    reference_lookup = {
        sid: idx for idx, sid in enumerate(reference_ids)
    }

    common_ids = [
        sid
        for sid in dataset_ids
        if sid in translation_lookup and sid in reference_lookup
    ][:n_max]

    if not common_ids:
        raise ValueError("No shared target/reference/translation sample IDs were found.")

    translation_values = to_numpy(recon_dict[translation_key])
    reference_values = to_numpy(recon_dict[reference_output_key])

    original_sum = None
    reference_sum = None
    translation_sum = None

    for sample_id in common_ids:
        original = extract_dataset_array(
            dataset,
            dataset_lookup[sample_id],
        )
        reference = np.asarray(
            reference_values[reference_lookup[sample_id]]
        ).squeeze().astype(np.float32)
        translation = np.asarray(
            translation_values[translation_lookup[sample_id]]
        ).squeeze().astype(np.float32)

        if original_sum is None:
            original_sum = np.zeros_like(original, dtype=np.float32)
            reference_sum = np.zeros_like(reference, dtype=np.float32)
            translation_sum = np.zeros_like(translation, dtype=np.float32)

        original_sum += original
        reference_sum += reference
        translation_sum += translation

    n = len(common_ids)

    return (
        original_sum / n,
        reference_sum / n,
        translation_sum / n,
        common_ids,
    )


try:
    mean_original, mean_reference, mean_translation, mean_map_ids = (
        compute_mean_target_outputs(
            result,
            target_modality=to_key,
            n_max=MEAN_MAP_MAX_SAMPLES,
        )
    )

    mean_translation_error = np.abs(
        mean_original - mean_translation
    )

    d_mid, h_mid, w_mid = [
        size // 2
        for size in mean_original.shape
    ]

    views = [
        (
            "Axial",
            lambda volume: volume[d_mid, :, :],
        ),
        (
            "Coronal",
            lambda volume: volume[:, h_mid, :],
        ),
        (
            "Sagittal",
            lambda volume: volume[:, :, w_mid],
        ),
    ]

    fig, axes = plt.subplots(
        3,
        4,
        figsize=(14, 11),
        constrained_layout=True,
    )

    intensity_values = np.concatenate(
        [
            mean_original.ravel(),
            mean_reference.ravel(),
            mean_translation.ravel(),
        ]
    )
    vmin, vmax = np.percentile(
        intensity_values,
        [1, 99],
    )

    for row, (view_name, slicer) in enumerate(views):
        panels = [
            ("Mean original", slicer(mean_original), "gray", vmin, vmax),
            ("Mean reference", slicer(mean_reference), "gray", vmin, vmax),
            ("Mean translation", slicer(mean_translation), "gray", vmin, vmax),
            (
                "Mean |translation error|",
                slicer(mean_translation_error),
                "magma",
                None,
                None,
            ),
        ]

        for col, (title, image, cmap, low, high) in enumerate(panels):
            axes[row, col].imshow(
                image,
                cmap=cmap,
                origin="lower",
                vmin=low,
                vmax=high,
            )
            axes[row, col].axis("off")

            if row == 0:
                axes[row, col].set_title(title)

            if col == 0:
                axes[row, col].set_ylabel(view_name)

    fig.suptitle(
        f"Mean target outputs: {from_key} -> {to_key} "
        f"(n={len(mean_map_ids)})",
        fontsize=14,
    )

    fig.savefig(
        report_dir / "mean_target_reference_translation_error.pdf",
        bbox_inches="tight",
    )
    plt.show()

except Exception as exc:
    print(f"Mean target translation maps unavailable: {exc}")


## Loss and latent-space visualizations

The inherited XModalix visualizations remain useful for XModalix3D because they operate on loss values and latent vectors rather than directly on 3D voxel arrays.

The 2D latent-space plot is qualitative: apparent overlap in a UMAP/PCA-style projection does **not** prove modality invariance. Quantitative alignment is evaluated separately below.


In [ ]:
# Only show loss plots that have non-zero values.

vis = xmodalix3d_loaded.visualizer

loss_df = vis._make_loss_format(
    result=result,
    config=xmodalix3d_loaded.config,
)

# Same XModalix filtering as visualize():
loss_df = loss_df[
    ~loss_df["Loss Term"].str.startswith(
        tuple(result.datasets.train.datasets.keys())
    )
]

# Remove loss terms that are entirely zero / NaN
valid_terms = [term
    for term in loss_df["Loss Term"].unique()
    if (loss_df.loc[loss_df["Loss Term"] == term, "Loss Value"].notna().any()
        and
        (loss_df.loc[loss_df["Loss Term"] == term, "Loss Value"] != 0).any()
    )
]

loss_df = loss_df[loss_df["Loss Term"].isin(valid_terms)]

# Generate and cache the filtered absolute plot
vis.plots["loss_absolute"] = vis._make_loss_plot(
    df_plot=loss_df,
    plot_type="absolute",
)

for plot_type in ["absolute", "relative"]:
    vis.show_loss(plot_type=plot_type)
    plt.gcf().savefig(
       report_dir / f"loss_{plot_type}.pdf",
       bbox_inches="tight",
   )

### Modality-wise latent-space visualization

#### Valid (top) and test (bottom) scatter plot & valid ridgeline plot

In [ ]:
for split in ["valid", "test"]:
    
    epoch = resolve_epoch(result.latentspaces, split)

    xmodalix3d_loaded.visualizer.show_latent_space(
        result=result,
        plot_type="2D-scatter",
        epoch=epoch,
        split=split,
    )

    plt.gcf().savefig(
        report_dir / f"latent_2d_scatter_{split}.pdf",
        bbox_inches="tight",
    )


valid_epoch = resolve_epoch(result.latentspaces, "valid")

xmodalix3d_loaded.visualizer.show_latent_space(
    result=result,
    plot_type="Ridgeline",
    epoch=valid_epoch,
    split="valid",
)

plt.gcf().savefig(
    report_dir / "latent_ridgeline_valid.pdf",
    bbox_inches="tight",
)

#### Scatter- & Ridgeline plot (all samples)

In [ ]:
xmodalix3d_loaded.visualizer.show_latent_space(
    result=result,
    plot_type="2D-scatter",
)
plt.gcf().savefig(
    report_dir / "latent_2d_scatter_all.pdf",
    dpi=300,
    bbox_inches="tight",
)

xmodalix3d_loaded.visualizer.show_latent_space(
    result=result,
    plot_type="Ridgeline",
)
plt.gcf().savefig(
    report_dir / "latent_ridgeline_all.pdf",
    dpi=300,
    bbox_inches="tight",
)

## Quantitative cross-modal alignment

The two XModalix3D-specific evaluation methods are used here.

### Modality linear probe

A logistic-regression classifier is trained on **frozen** embeddings to predict modality identity. Cross-validation is grouped by patient ID, so embeddings belonging to the same patient cannot be split between probe training and validation folds.

For $K$ modalities, balanced-accuracy chance level is $\frac{1}{K}$. Lower probe accuracy therefore indicates stronger modality invariance.

### Matched/unmatched analysis

For each modality pair, the mean latent distance between **matched measurements of the same patient** is compared with the mean distance between **different patients**:

$$
R = \frac{\overline d_{\mathrm{matched}}}{\overline d_{\mathrm{unmatched}}}
$$

Values below 1 indicate patient-specific cross-modal correspondence; smaller values indicate stronger correspondence.


In [ ]:
required_alignment_methods = [
    "modality_linear_probe",
    "matched_unmatched_analysis",
]

required_alignment_visualizations = [
    "show_modality_linear_probe",
    "show_matched_unmatched_analysis",
]

for method in required_alignment_methods:
    if not hasattr(xmodalix3d_loaded.evaluator, method):
        raise AttributeError(
            f"XModalix3DEvaluator is missing required method: {method}"
        )

for method in required_alignment_visualizations:
    if not hasattr(xmodalix3d_loaded.visualizer, method):
        raise AttributeError(
            f"XModal3DVisualizer is missing required method: {method}"
        )

print("Alignment evaluator and visualizer methods are available.")


### Final validation/test modality linear-probe results

In [ ]:
probe_summaries = []
probe_confusions = {}
representation = "z"

for split in ['valid','test']:
    
    store =  result.latentspaces
    epoch = resolve_epoch(store, split)

    summary_df, confusion_df = (
        xmodalix3d_loaded.evaluator.modality_linear_probe(
            result=result,
            representation=representation,
            split=split,
            epoch=epoch,
            n_splits=LINEAR_PROBE_CV_SPLITS,
            C=LINEAR_PROBE_C,
        )
    )

    summary_df = summary_df.copy()
    summary_df["display_epoch"] = (
            "final" if epoch == -1 else epoch + 1
    )

    probe_summaries.append(summary_df)
    probe_confusions[(split, representation)] = confusion_df

    confusion_df.to_csv(
        report_dir
        / f"modality_linear_probe_confusion_{split}_{representation}.csv"
    )

    xmodalix3d_loaded.visualizer.show_modality_linear_probe(
        summary_df=summary_df.drop(columns=["display_epoch"]),
        confusion_df=confusion_df,
    )

    plt.gcf().savefig(
        report_dir
        / f"modality_linear_probe_{split}_{representation}.pdf",
        bbox_inches="tight",
    )

probe_summary_all = (
    pd.concat(probe_summaries, ignore_index=True)
    if probe_summaries
    else pd.DataFrame()
)

display(probe_summary_all)

if not probe_summary_all.empty:
    probe_summary_all.to_csv(
        report_dir / "modality_linear_probe_summary.csv",
        index=False,
    )

### Final validation/test matched/unmatched analysis

In [ ]:
alignment_summaries = []
alignment_distance_tables = {}
representation = "z"

for split in ['valid', 'test']:
    
    store =  result.latentspaces
    epoch = resolve_epoch(store, split)

    summary_df, distances_df = (
        xmodalix3d_loaded.evaluator.matched_unmatched_analysis(
            result=result,
            representation=representation,
            split=split,
            epoch=epoch,
            distance_metric="euclidean",
            return_distances=True,
        )
    )

    summary_df = summary_df.copy()
    summary_df["display_epoch"] = ("final" if epoch == -1 else epoch + 1)

    alignment_summaries.append(summary_df)
    alignment_distance_tables[(split, representation)] = distances_df

    summary_df.to_csv(
        report_dir
        / f"matched_unmatched_summary_{split}_{representation}.csv",
        index=False,
    )

    distances_df.to_csv(
        report_dir
        / f"matched_unmatched_distances_{split}_{representation}.csv",
        index=False,
    )

    xmodalix3d_loaded.visualizer.show_matched_unmatched_analysis(
        summary_df=summary_df.drop(columns=["display_epoch"]),
        distances_df=distances_df,
    )

    plt.gcf().savefig(
        report_dir
        / f"matched_unmatched_{split}_{representation}.pdf",
        bbox_inches="tight",
    )


alignment_summary_all = (
    pd.concat(alignment_summaries, ignore_index=True)
    if alignment_summaries
    else pd.DataFrame()
)

display(alignment_summary_all)

if not alignment_summary_all.empty:
    alignment_summary_all.to_csv(
        report_dir / "matched_unmatched_summary_all.csv",
        index=False,
    )

### Alignment across validation checkpoints

The final alignment numbers alone can hide changes during joint training. This section evaluates the stored validation checkpoints and plots:

1. modality linear-probe balanced accuracy over joint-training epochs;
2. mean matched/unmatched distance ratio over joint-training epochs.

This can reveal, for example, whether patient-specific alignment improves early and then deteriorates again later in training.


In [ ]:
z_epochs = set(stored_epochs_for_split(result.latentspaces, "valid"))
id_epochs = set(stored_epochs_for_split(result.sample_ids, "valid"))

validation_alignment_epochs = sorted(
    z_epochs & id_epochs
)

trajectory_rows = []

for epoch in validation_alignment_epochs:
    for representation in ALIGNMENT_REPRESENTATIONS:
        try:
            probe_df, _ = (
                xmodalix3d_loaded.evaluator.modality_linear_probe(
                    result=result,
                    representation=representation,
                    split="valid",
                    epoch=epoch,
                    n_splits=LINEAR_PROBE_CV_SPLITS,
                    C=LINEAR_PROBE_C,
                )
            )

            pair_df = (
                xmodalix3d_loaded.evaluator.matched_unmatched_analysis(
                    result=result,
                    representation=representation,
                    split="valid",
                    epoch=epoch,
                    distance_metric="euclidean",
                    return_distances=False,
                )
            )

            trajectory_rows.append(
                {
                    "epoch": epoch,
                    "display_epoch": epoch + 1,
                    "representation": representation,
                    "linear_probe_balanced_accuracy": float(
                        probe_df[
                            "linear_probe_balanced_accuracy"
                        ].iloc[0]
                    ),
                    "chance_level": float(
                        probe_df["chance_level"].iloc[0]
                    ),
                    "mean_matched_unmatched_ratio": float(
                        pair_df[
                            "matched_unmatched_ratio"
                        ].mean()
                    ),
                }
            )

        except Exception as exc:
            print(
                f"Could not evaluate validation epoch "
                f"{epoch + 1}, {representation}: {exc}"
            )

alignment_trajectory = pd.DataFrame(
    trajectory_rows
)

display(alignment_trajectory)

if not alignment_trajectory.empty:
    alignment_trajectory.to_csv(
        report_dir / "alignment_validation_trajectory.csv",
        index=False,
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(13, 5),
        constrained_layout=True,
    )

    sns.lineplot(
        data=alignment_trajectory,
        x="display_epoch",
        y="linear_probe_balanced_accuracy",
        hue="representation",
        marker="o",
        ax=axes[0],
    )

    chance_values = alignment_trajectory[
        "chance_level"
    ].dropna()

    if not chance_values.empty:
        axes[0].axhline(
            chance_values.iloc[0],
            linestyle="--",
            label="Chance",
        )

    axes[0].set_title(
        "Validation modality linear probe"
    )
    axes[0].set_xlabel(
        "Joint-training epoch"
    )
    axes[0].set_ylabel(
        "Balanced accuracy"
    )

    sns.lineplot(
        data=alignment_trajectory,
        x="display_epoch",
        y="mean_matched_unmatched_ratio",
        hue="representation",
        marker="o",
        ax=axes[1],
    )

    axes[1].axhline(
        1.0,
        linestyle="--",
        label="No patient-specific alignment",
    )

    axes[1].set_title(
        "Validation patient-specific alignment"
    )
    axes[1].set_xlabel(
        "Joint-training epoch"
    )
    axes[1].set_ylabel(
        "Mean matched / unmatched distance ratio"
    )

    fig.savefig(
        report_dir / "alignment_validation_trajectory.pdf",
        ,
        bbox_inches="tight",
    )
    plt.show()
else:
    print(
        "No validation checkpoint trajectory could be computed. "
        "This is expected if intermediate latent dynamics were not saved."
    )

### Train/validation loss-gap table

The train/validation gap is calculated for every stored loss term. This is useful for inspecting whether reconstruction, variational, adversarial or paired losses begin to diverge between training and validation.


In [ ]:
if losses_long.empty:
    loss_gap_df = pd.DataFrame()
    print("Loss-gap table unavailable because no stored losses were found.")
else:
    pivot = (
        losses_long
        .pivot_table(
            index=["epoch", "display_epoch", "loss_term"],
            columns="split",
            values="value",
            aggfunc="first",
        )
        .reset_index()
    )

    if {"train", "valid"}.issubset(pivot.columns):
        loss_gap_df = pivot.dropna(
            subset=["train", "valid"]
        ).copy()

        loss_gap_df[
            "valid_minus_train"
        ] = (
            loss_gap_df["valid"]
            - loss_gap_df["train"]
        )

        loss_gap_df[
            "valid_div_train"
        ] = (
            loss_gap_df["valid"]
            / loss_gap_df["train"].replace(0, np.nan)
        )

        display(
            loss_gap_df
            .sort_values(["loss_term", "epoch"])
            .groupby("loss_term", as_index=False)
            .tail(1)
        )

        loss_gap_df.to_csv(
            report_dir / "loss_train_valid_gap.csv",
            index=False,
        )
    else:
        loss_gap_df = pd.DataFrame()
        print("Both train and valid losses are required for the gap table.")

## Compact final report summary

The table below collects the principal quantitative XModalix3D alignment results. It is intentionally compact: detailed probe confusion matrices, pairwise matched/unmatched tables, individual distance distributions, reconstruction metrics and loss curves are exported separately to `REPORT_DIR`.


In [ ]:
summary_parts = []

if not probe_summary_all.empty:
    probe_compact = probe_summary_all[
        [
            "split",
            "representation",
            "linear_probe_balanced_accuracy",
            "chance_level",
        ]
    ].copy()

    probe_compact = probe_compact.rename(
        columns={
            "linear_probe_balanced_accuracy": "probe_balanced_accuracy",
        }
    )

    summary_parts.append(
        ("probe", probe_compact)
    )

if not alignment_summary_all.empty:
    ratio_compact = (
        alignment_summary_all
        .groupby(
            ["split", "representation"],
            as_index=False,
        )
        .agg(
            mean_matched_unmatched_ratio=(
                "matched_unmatched_ratio",
                "mean",
            ),
            min_matched_unmatched_ratio=(
                "matched_unmatched_ratio",
                "min",
            ),
            max_matched_unmatched_ratio=(
                "matched_unmatched_ratio",
                "max",
            ),
        )
    )

    summary_parts.append(
        ("ratio", ratio_compact)
    )

if summary_parts:
    final_report_summary = summary_parts[0][1]

    for _, frame in summary_parts[1:]:
        final_report_summary = final_report_summary.merge(
            frame,
            on=["split", "representation"],
            how="outer",
        )

    display(final_report_summary)

    final_report_summary.to_csv(
        report_dir / "final_alignment_summary.csv",
        index=False,
    )
else:
    final_report_summary = pd.DataFrame()
    print("No final alignment summary could be generated.")

print("\nReport files written to:")
print(report_dir)

## Interpretation guide

For the two primary alignment diagnostics:

- **Modality linear probe:** lower is better for modality invariance. With three modalities, chance balanced accuracy is \(1/3 \approx 0.333\). High probe accuracy means modality identity remains linearly recoverable.
- **Matched/unmatched ratio:** lower is better for patient-specific correspondence. A ratio below 1 means matched patients are closer across modalities than unmatched patients.

These two properties should not be conflated. A model can preserve strong patient-specific correspondence while still retaining enough modality-specific structure for a linear classifier to identify the originating modality.

Unlike the Imagix3D report, this notebook does not currently include latent-activity or latent-traversal analyses because those methods belong to the separate `Imagix3DEvaluator` extension and have not yet been integrated into `XModalix3DEvaluator`.
